# HAIKUTHON 학습 (Colab GPU) — kimth v1

**v1에서 바뀐 점** (코드: `starter_kit_kimth_v1.zip`)
- **검증 4,000장**: 검증 프롬프트 800개가 검증+학습 이미지 4,000장에서 정답을 찾습니다. 형제 그룹이 갖춰져 공개 점수와 비슷한 난이도가 됩니다. 이전 800장 기준보다 점수가 크게 낮게 나오는 것이 정상입니다.
- **Top-K 묶음 재정렬**: SigLIP으로 Top-20을 뽑고, BLIP-2는 모든 질의의 1~5위 → 6~10위 → 11~15위 → 16~20위 순으로 채점합니다. 시간이 다 되면 끝낸 묶음까지만 씁니다. ViT-g 특징은 캐시해 이미지당 한 번만 계산합니다.
- **시간 예산 840초** (`load_model` 시작 기준, 이전 720초)

**시작 전에**
1. 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU** (T4 / L4 / A100) 선택
2. Google Drive에 `내 드라이브/haikuthon/` 폴더를 만들고 아래 두 파일을 올립니다.
   - `haikuthon-train.zip` — 학습 데이터 (이미지 4,000장 + prompts.jsonl, 약 3.65GB). `DATA_SOURCE = "hf"`이면 올리지 않아도 됩니다.
   - `starter_kit_kimth_v1.zip` — `train.py`, `sample_submission/model.py` 등 코드
3. 위에서부터 셀을 차례로 실행합니다. (메뉴 **런타임 → 모두 실행**도 됩니다)

결과(`submission.zip`, 학습 로그, `pipeline.json`)는 `내 드라이브/haikuthon/outputs/<날짜_시각>/`에 저장됩니다.

In [ ]:
# ---- 설정 ----
DRIVE_DIR = "/content/drive/MyDrive/haikuthon"   # ZIP을 올린 Drive 폴더
CODE_ZIP = "starter_kit_kimth_v2.zip"           # Drive에 올린 코드 ZIP 이름

# "drive": Drive에 올린 haikuthon-train.zip 사용
# "hf"   : Hugging Face에서 같은 공개 데이터를 바로 받기 (업로드 없이 1~2분)
DATA_SOURCE = "drive"

# train.py 옵션. --rerank-k는 재정렬할 최대 후보 수(5개씩 묶어 시간 안에서 최대한).
# 검증은 기본으로 4,000장 후보(--val-gallery all). 예전 800장 방식은 --val-gallery valid.
# 예) 1단계만 빠르게: "--epochs 1 --no-rerank"
TRAIN_ARGS = "--epochs 1 --batch-size 16 --rerank-epochs 2 --rerank-k 20"

# 학습이 끝난 뒤 테스트 1/4 규모(프롬프트 2,000 × 이미지 4,000)로 추론 시간을 잴지 (맨 아래 7번 셀, 몇 분 걸림)
RUN_TIMING_CHECK = False

## 1. GPU 확인

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "GPU가 없습니다. 런타임 → 런타임 유형 변경에서 GPU를 선택한 뒤 다시 실행하세요."
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

## 2. Drive 연결 → 코드·데이터를 Colab 로컬 디스크로 복사

Drive에서 바로 읽으면 느려서 `/content`로 복사한 뒤 압축을 풉니다.

In [ ]:
import shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path(DRIVE_DIR)
WORK = Path("/content/starterkit")
DATA = WORK / "haikuthon-train"
WORK.mkdir(parents=True, exist_ok=True)

# 코드: 매번 덮어써서 Drive에 새로 올린 코드가 반영되게 합니다.
code_zip = DRIVE / CODE_ZIP
assert code_zip.is_file(), f"{code_zip}가 없습니다. Drive에 {CODE_ZIP}을 올려 주세요."
with zipfile.ZipFile(code_zip) as z:
    z.extractall(WORK)
print("코드 압축 해제:", sorted(p.name for p in WORK.iterdir()))

def data_ready():
    return (DATA / "prompts.jsonl").is_file() and len(list((DATA / "images").glob("*.png"))) == 4000

if data_ready():
    print("데이터가 이미 준비되어 있습니다.")
elif DATA_SOURCE == "drive":
    data_zip = DRIVE / "haikuthon-train.zip"
    assert data_zip.is_file(), f"{data_zip}가 없습니다. Drive에 haikuthon-train.zip을 올려 주세요."
    local_zip = Path("/content/haikuthon-train.zip")
    print(f"Drive → 로컬 복사 중 ({data_zip.stat().st_size / 1e9:.2f} GB)...")
    shutil.copy(data_zip, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(WORK)            # ZIP 안의 haikuthon-train/ 폴더가 그대로 풀립니다.
    local_zip.unlink()
elif DATA_SOURCE == "hf":
    from huggingface_hub import snapshot_download
    snapshot_download("KimWooJin/26-haikuthon-train", repo_type="dataset",
                      allow_patterns=["prompts.jsonl", "images/*.png"], local_dir=DATA)
else:
    raise ValueError("DATA_SOURCE는 'drive' 또는 'hf'입니다.")

n_prompts = sum(1 for line in open(DATA / "prompts.jsonl", encoding="utf-8") if line.strip())
n_images = len(list((DATA / "images").glob("*.png")))
print(f"프롬프트 {n_prompts}개 / 이미지 {n_images}장")
assert n_prompts == n_images == 4000

## 3. 패키지 설치

`requirements.txt`의 버전을 그대로 설치하되, **torch/torchvision은 Colab에 깔린 CUDA 빌드를 그대로** 씁니다(드라이버와 맞춰져 있음).
학습은 별도 프로세스로 실행하므로 "런타임 다시 시작" 안내가 떠도 무시해도 됩니다.

In [ ]:
!pip install -q transformers==5.22.2 sentencepiece==0.2.3 protobuf==7.36.2 safetensors==0.8.0 pillow==13.0.0 tqdm==4.70.1
!python -c "import torch, transformers, PIL; print('torch', torch.__version__, '| transformers', transformers.__version__, '| pillow', PIL.__version__, '| cuda', torch.cuda.is_available())"

## 4. 학습

1단계 SigLIP 학습 → 2단계 BLIP-2 재정렬기 학습 → 검증(4,000장 후보) → `sample_submission/`에 저장 → `/content/submission.zip` 생성.
처음 실행할 때 SigLIP(약 0.8GB)과 BLIP-2(약 4.6GB)를 내려받습니다. T4 기준 1시간 남짓 걸립니다(검증 후보가 4,000장이라 이전보다 몇 분 더 걸립니다).
무료 Colab은 탭을 오래 비워 두면 연결이 끊기니 탭을 열어 두세요.

로그에서 볼 것
- `[2단계 …] 1단계 후보 재현율 (재정렬 상한) R@5 … | R@10 … | R@15 … | R@20 …` — 재정렬이 도달할 수 있는 상한
- `재정렬 6~10위 완료: …초` — 묶음별 소요 시간
- `최적 w=…` — 이 값이 `pipeline.json`의 `rerank_weight`로 저장됩니다

In [ ]:
import os
os.environ["HF_HOME"] = "/content/hf_cache"   # 모델 캐시는 로컬 디스크에
os.environ["PYTHONUNBUFFERED"] = "1"
%cd /content/starterkit
!rm -f /content/submission.zip
!python train.py --data-dir haikuthon-train --device cuda --output /content/submission.zip {TRAIN_ARGS} 2>&1 | tee /content/train.log
assert Path("/content/submission.zip").is_file(), "학습이 끝까지 가지 못했습니다. 위 로그의 오류를 확인하세요."
print("완료:", f"{Path('/content/submission.zip').stat().st_size / 1e9:.2f} GB")

## 5. 학습 결과 요약

로그에서 에포크별 점수와 저장 결과만 모아 봅니다.

In [ ]:
KEYS = ("[1단계]", "[2단계", "  1단계만", "  재정렬만", "  최적 w", "  저장", "1단계 가중치 저장")
for line in open("/content/train.log", encoding="utf-8", errors="replace"):
    line = line.rstrip().split("\r")[-1]
    if line.startswith(KEYS):
        print(line)
pipeline_file = WORK / "sample_submission" / "pipeline.json"
if pipeline_file.is_file():
    print("\npipeline.json:", pipeline_file.read_text(encoding="utf-8"))

## 6. 결과를 Drive에 저장

In [ ]:
import time
out = DRIVE / "outputs" / time.strftime("%m%d_%H%M")
out.mkdir(parents=True, exist_ok=True)
shutil.copy("/content/train.log", out / "train.log")
pipeline = WORK / "sample_submission" / "pipeline.json"
if pipeline.is_file():
    shutil.copy(pipeline, out / "pipeline.json")
print("submission.zip을 Drive로 복사 중...")
shutil.copy("/content/submission.zip", out / "submission.zip")
print("저장 위치:", out)
for p in sorted(out.iterdir()):
    print(f"  {p.name:20s} {p.stat().st_size / 1e6:10.1f} MB")

## 7. (선택) 추론 시간 점검

`RUN_TIMING_CHECK = True`일 때만 실행합니다. 방금 학습한 제출 모델로 테스트와 같은 1:2 비율의 1/4 규모(프롬프트 2,000개 × 이미지 4,000장)에서 `predict()`를 돌립니다.
테스트 시간은 대략 `로드 + predict × 4`입니다. Colab GPU와 평가 서버 GPU가 다를 수 있으니 절대 시간보다 묶음별 시간 비율을 참고하세요.
이 규모에서는 ViT-g 캐시가 약 2.9GB라 GPU에 다 들어가지만, 테스트(16,000장)에서는 약 11.6GB가 필요합니다.

In [ ]:
import subprocess, sys
if RUN_TIMING_CHECK:
    script = r"""
import sys, time
from pathlib import Path
sys.path.insert(0, "/content/starterkit")
from train import load_and_split
from sample_submission.model import load_model, predict

train, valid = load_and_split(Path("/content/starterkit/haikuthon-train"), 42)
rows = valid + train
queries = [{"prompt_id": r["id"], "prompt": r["prompts"]} for r in rows[::2]]        # 2,000개
candidates = [{"image_id": r["id"], "image_path": r["image_path"]} for r in rows]     # 4,000장
t0 = time.monotonic()
model = load_model("/content/starterkit/sample_submission", "cuda")
t1 = time.monotonic()
predict(model, queries, candidates)
t2 = time.monotonic()
budget = model["config"]["time_budget_sec"]
print(f"로드 {t1 - t0:.0f}초 / predict {t2 - t1:.0f}초 (rerank_k {model['config']['rerank_k']})")
print(f"테스트(프롬프트 8,000 × 이미지 16,000) 추정: {(t1 - t0) + 4 * (t2 - t1):.0f}초 / 시간 예산 {budget}초")
"""
    Path("/content/timing_check.py").write_text(script, encoding="utf-8")
    result = subprocess.run([sys.executable, "/content/timing_check.py"], cwd="/content/starterkit",
                            capture_output=True, text=True)
    print("\n".join(line for line in (result.stdout + result.stderr).splitlines()
                    if "Loading weights" not in line and "token_id must be" not in line))
else:
    print("RUN_TIMING_CHECK = False: 건너뜁니다.")